# Poke Agent Unified Run

Run this notebook end-to-end.

- On Kaggle/Linux with CABT `cg-lib` available, it can generate rollout data.
- On this Mac, it uses existing rollout JSONL data and trains with Torch on Apple Silicon MPS.
- It does not submit to the competition leaderboard.


In [33]:
from __future__ import annotations

import glob
import hashlib
import json
import os
import random
import sys
from pathlib import Path

import numpy as np
import torch
from tqdm.auto import tqdm

ROOT = Path.cwd()
if not (ROOT / "requirements.txt").exists() and (ROOT.parent / "requirements.txt").exists():
    ROOT = ROOT.parent
os.chdir(ROOT)

print("repo", ROOT)
print("python", sys.version.split()[0])
print("torch", torch.__version__)


repo /home/inzi/poke-bot-agent
python 3.11.15
torch 2.12.1+cu130


In [34]:
# Top-level config: edit here or override with environment variables.
def env_int(name: str, default: int) -> int:
    return int(os.environ.get(name, str(default)))


def env_float(name: str, default: float) -> float:
    return float(os.environ.get(name, str(default)))


CONFIG = {
    "agent_deck_path": ROOT / os.environ.get(
        "AGENT_DECK_PATH",
        "decks/competitive/high_performing/2026-05_regional-campinas-2026_4th_dragapult-dudunsparce.csv",
    ),
    "data_candidates": [
        ROOT / os.environ.get("PRIMARY_ROLLOUT_DATA", "data/mac-rollouts-100k-fullstate.jsonl"),
        ROOT / "data/mac-rollouts-10k.jsonl",
        ROOT / "data/notebook_rollouts.jsonl",
        ROOT / "data/kaggle-output/data/cabt_rollouts.jsonl",
        ROOT / "data/deckpool-smoke.jsonl",
        ROOT / "data/container-mp-smoke.jsonl",
        ROOT / "data/container-smoke.jsonl",
    ],
    "generated_path": ROOT / os.environ.get("CABT_GENERATED_PATH", "data/notebook_rollouts.jsonl"),
    "competition_results_path": ROOT / os.environ.get("COMPETITION_RESULTS_PATH", "data/competition-results.jsonl"),
    "transition_classes": env_int("TRANSITION_CLASSES", 8),
    "state_hash_dim": env_int("STATE_HASH_DIM", 256),
    "window_size": env_int("WINDOW_SIZE", 128),
    "model": {
        "d_model": env_int("MODEL_D_MODEL", 512),
        "heads": env_int("MODEL_HEADS", 8),
        "layers": env_int("MODEL_LAYERS", 8),
        "ff": env_int("MODEL_FF", env_int("MODEL_D_MODEL", 512) * 4),
        "dropout": env_float("MODEL_DROPOUT", 0.1),
        "learning_rate": env_float("LEARNING_RATE", 3e-4),
        "weight_decay": env_float("WEIGHT_DECAY", 1e-2),
    },
    "loss": {
        "value": env_float("LOSS_VALUE_WEIGHT", 1.0),
        "policy": env_float("LOSS_POLICY_WEIGHT", 0.35),
        "dynamics": env_float("LOSS_DYNAMICS_WEIGHT", 0.15),
        "entropy": env_float("LOSS_ENTROPY_WEIGHT", 0.01),
        "uncertainty": env_float("LOSS_UNCERTAINTY_WEIGHT", 0.02),
    },
    "training": {
        "epochs": env_int("TRAIN_EPOCHS", 500),
        "patience": env_int("EARLY_STOP_PATIENCE", 10),
        "min_delta": env_float("EARLY_STOP_MIN_DELTA", 1e-5),
        "print_every": env_int("TRAIN_PRINT_EVERY", 100),
        "batch_size": env_int("BATCH_SIZE", 256),
    },
}

CONFIG


{'data_candidates': [PosixPath('/home/inzi/poke-bot-agent/data/mac-rollouts-100k-fullstate.jsonl'),
  PosixPath('/home/inzi/poke-bot-agent/data/mac-rollouts-10k.jsonl'),
  PosixPath('/home/inzi/poke-bot-agent/data/notebook_rollouts.jsonl'),
  PosixPath('/home/inzi/poke-bot-agent/data/kaggle-output/data/cabt_rollouts.jsonl'),
  PosixPath('/home/inzi/poke-bot-agent/data/deckpool-smoke.jsonl'),
  PosixPath('/home/inzi/poke-bot-agent/data/container-mp-smoke.jsonl'),
  PosixPath('/home/inzi/poke-bot-agent/data/container-smoke.jsonl')],
 'generated_path': PosixPath('/home/inzi/poke-bot-agent/data/notebook_rollouts.jsonl'),
 'competition_results_path': PosixPath('/home/inzi/poke-bot-agent/data/competition-results.jsonl'),
 'transition_classes': 8,
 'state_hash_dim': 256,
 'window_size': 128,
 'model': {'d_model': 512,
  'heads': 8,
  'layers': 8,
  'ff': 2048,
  'dropout': 0.1,
  'learning_rate': 0.0003,
  'weight_decay': 0.01},
 'loss': {'value': 1.0,
  'policy': 0.35,
  'dynamics': 0.15,
  

In [35]:
def torch_device() -> torch.device:
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

DEVICE = torch_device()
print("device", DEVICE)


device cuda


In [36]:
def find_cg_lib() -> str | None:
    candidates: list[str] = []
    if os.environ.get("CG_LIB_PATH"):
        candidates.append(os.environ["CG_LIB_PATH"])
    candidates.extend(glob.glob("/kaggle/input/**/cg-lib", recursive=True))
    candidates.extend(glob.glob(str(ROOT / "kaggle/input/**/cg-lib"), recursive=True))
    return candidates[0] if candidates else None

CG_LIB_PATH = find_cg_lib()
CG_AVAILABLE = False
CG_ERROR = None
if CG_LIB_PATH:
    sys.path.append(CG_LIB_PATH)
    try:
        from cg.game import battle_finish, battle_select, battle_start
        from cg.api import to_observation_class
        CG_AVAILABLE = True
    except Exception as exc:
        CG_ERROR = repr(exc)

print("cg_lib_path", CG_LIB_PATH)
print("cg_available", CG_AVAILABLE)
if CG_ERROR:
    print("cg_error", CG_ERROR)


cg_lib_path /home/inzi/poke-bot-agent/kaggle/input/cg-lib
cg_available True


In [37]:
SAMPLE_DECK = [
    119, 119, 119, 119, 120, 120, 120, 120, 121, 121, 121,
    305, 305, 66, 66, 112, 112, 235, 1071, 140, 1227,
    1227, 1227, 1227, 1182, 1182, 1182, 1198, 1198, 1240, 1213,
    1086, 1086, 1086, 1086, 1152, 1152, 1152, 1152, 1121, 1121,
    1121, 1121, 1120, 1120, 1120, 1120, 1097, 1097, 1080, 1260,
    1260, 2, 2, 2, 5, 5, 5, 7, 7,
]

def read_deck() -> tuple[list[int], Path]:
    candidates = [
        CONFIG["agent_deck_path"],
        ROOT / "decks/submission.csv",
        ROOT / "submission/deck.csv",
        ROOT / "deck.csv",
        Path("/kaggle_simulations/agent/deck.csv"),
    ]
    for path in candidates:
        if path.exists():
            deck = [int(line.strip()) for line in path.read_text().splitlines() if line.strip()]
            if len(deck) != 60:
                raise ValueError(f"{path} must contain 60 card IDs")
            return deck, path
    return SAMPLE_DECK, Path("<sample>")

DECK, DECK_SOURCE = read_deck()
print("deck cards", len(DECK))
print("deck source", DECK_SOURCE)


deck cards 60


In [38]:
def random_agent(obs_dict: dict) -> list[int]:
    obs = to_observation_class(obs_dict)
    options = list(range(len(obs.select.option)))
    return random.sample(options, min(obs.select.maxCount, len(options)))


def features_from_observation(obs: dict) -> list[float]:
    current = obs.get("current") or {}
    players = current.get("players") or [{}, {}]
    p0 = players[0] if len(players) > 0 else {}
    p1 = players[1] if len(players) > 1 else {}
    select = obs.get("select") or {}
    return [
        float(current.get("turn", 0)),
        float(current.get("yourIndex", 0)),
        float(p0.get("deckCount", 0)),
        float(p0.get("handCount", 0)),
        float(len(p0.get("bench", []))),
        float(p1.get("deckCount", 0)),
        float(p1.get("handCount", 0)),
        float(len(p1.get("bench", []))),
        float(len(select.get("option", []))),
        float(select.get("maxCount", 0)),
    ]


def play_episode(episode: int, max_steps: int = 300) -> list[dict]:
    rows = []
    obs, start_data = battle_start(DECK, DECK)
    if start_data.errorPlayer >= 0:
        raise ValueError(f"deck error type={start_data.errorType} player={start_data.errorPlayer}")
    try:
        step = 0
        while obs["current"]["result"] < 0 and step < max_steps:
            rows.append({
                "episode": episode,
                "step": step,
                "features": features_from_observation(obs),
                "player": int(obs["current"]["yourIndex"]),
            })
            obs = battle_select(random_agent(obs))
            step += 1
        result = int(obs["current"]["result"])
        for row in rows:
            row["value"] = 0.0 if result == 2 else (1.0 if row["player"] == result else -1.0)
        return rows
    finally:
        battle_finish()


In [39]:
GENERATE_EPISODES = int(os.environ.get("CABT_EPISODES", "3" if CG_AVAILABLE else "0"))
GENERATED_PATH = CONFIG["generated_path"]

if CG_AVAILABLE and GENERATE_EPISODES > 0:
    GENERATED_PATH.parent.mkdir(parents=True, exist_ok=True)
    rows = []
    for episode in range(GENERATE_EPISODES):
        rows.extend(play_episode(episode))
    with GENERATED_PATH.open("w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, separators=(",", ":")) + "\n")
    print(f"generated {len(rows)} rows -> {GENERATED_PATH}")
else:
    print("skipping CABT generation in this runtime")


generated 203 rows -> /home/inzi/poke-bot-agent/data/notebook_rollouts.jsonl


In [40]:
DATA_CANDIDATES = CONFIG["data_candidates"]

TRANSITION_CLASSES = CONFIG["transition_classes"]

def load_jsonl(path: Path) -> list[dict]:
    with path.open("r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]


STATE_HASH_DIM = CONFIG["state_hash_dim"]
WINDOW_SIZE = CONFIG["window_size"]

def stable_hash_index(text: str, size: int) -> int:
    digest = hashlib.blake2b(text.encode("utf-8"), digest_size=8).digest()
    return int.from_bytes(digest, "little") % size


def iter_state_items(value, prefix: str = ""):
    if isinstance(value, dict):
        for key in sorted(value):
            child_prefix = f"{prefix}.{key}" if prefix else str(key)
            yield from iter_state_items(value[key], child_prefix)
    elif isinstance(value, list):
        for idx, item in enumerate(value):
            child_prefix = f"{prefix}[{idx}]"
            yield from iter_state_items(item, child_prefix)
    else:
        yield prefix, value


def hashed_state_vector(observation, action=None) -> np.ndarray:
    vec = np.zeros(STATE_HASH_DIM, dtype=np.float32)
    payloads = [("obs", observation, 1.0), ("action", action, 0.5)]
    for label, payload, weight in payloads:
        if payload is None:
            continue
        for key, value in iter_state_items(payload):
            if isinstance(value, bool):
                token = f"{label}.{key}:bool"
                amount = 1.0 if value else -1.0
            elif isinstance(value, (int, float)):
                token = f"{label}.{key}:num"
                amount = float(np.tanh(float(value) / 100.0))
            elif value is None:
                token = f"{label}.{key}:none"
                amount = 1.0
            else:
                token = f"{label}.{key}={value}"
                amount = 1.0
            vec[stable_hash_index(token, STATE_HASH_DIM)] += weight * amount
    return vec


def combine_features(coarse: list[float], observation=None, action=None) -> np.ndarray:
    compact = np.array(coarse, dtype=np.float32)
    if observation is None:
        return compact
    return np.concatenate([compact, hashed_state_vector(observation, action)]).astype(np.float32)


def row_feature_vector(row: dict) -> np.ndarray:
    return combine_features(row["features"], row.get("observation"), row.get("action"))


def row_next_feature_vector(row: dict) -> np.ndarray:
    if "next_observation" in row and "next_features" in row:
        return combine_features(row["next_features"], row.get("next_observation"), None)
    return row_feature_vector(row)


def build_training_arrays(rows: list[dict]) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    by_episode: dict[int, list[dict]] = {}
    for row in rows:
        by_episode.setdefault(int(row["episode"]), []).append(row)
    for episode_rows in by_episode.values():
        episode_rows.sort(key=lambda row: int(row["step"]))

    xs = []
    values = []
    transition_targets = []
    next_features = []
    terminal_mask = []
    history_indices = []
    history_mask = []

    for episode_rows in by_episode.values():
        episode_indices = []
        for idx, row in enumerate(episode_rows):
            current_index = len(xs)
            features = row_feature_vector(row)
            value = float(row["value"])
            if "next_observation" in row and "next_features" in row:
                next_feature = row_next_feature_vector(row)
                is_terminal = float(row.get("terminal", False))
            elif idx + 1 < len(episode_rows):
                next_row = episode_rows[idx + 1]
                next_feature = row_feature_vector(next_row)
                is_terminal = 0.0
            else:
                next_feature = features.copy()
                is_terminal = 1.0

            if "action" in row:
                action_key = json.dumps(row["action"], sort_keys=True, separators=(",", ":"))
                transition_class = stable_hash_index(action_key, TRANSITION_CLASSES)
            elif is_terminal:
                transition_class = TRANSITION_CLASSES - 1
            else:
                delta = next_feature[: len(row["features"])] - features[: len(row["features"])]
                transition_class = int(abs(delta).argmax()) % TRANSITION_CLASSES

            context = (episode_indices + [current_index])[-WINDOW_SIZE:]
            pad_count = WINDOW_SIZE - len(context)
            history_indices.append(([-1] * pad_count) + context)
            history_mask.append(([0.0] * pad_count) + ([1.0] * len(context)))

            xs.append(features)
            values.append(value)
            transition_targets.append(transition_class)
            next_features.append(next_feature)
            terminal_mask.append(is_terminal)
            episode_indices.append(current_index)

    pad_index = len(xs)
    history_indices_np = np.array(history_indices, dtype=np.int64)
    history_indices_np[history_indices_np < 0] = pad_index

    return (
        np.stack(xs).astype(np.float32),
        np.array(values, dtype=np.float32),
        np.array(transition_targets, dtype=np.int64),
        np.stack(next_features).astype(np.float32),
        np.array(terminal_mask, dtype=np.float32),
        history_indices_np,
        np.array(history_mask, dtype=np.float32),
    )


DATA_PATH = next((path for path in DATA_CANDIDATES if path.exists()), None)
if DATA_PATH is None:
    print("No rollout data found. Using synthetic smoke data so Run All still completes.")
    rng = np.random.default_rng(7)
    x_np = rng.normal(size=(128, 10)).astype(np.float32)
    y_np = np.tanh(x_np[:, 0] * 0.1 + x_np[:, 2] * 0.03 - x_np[:, 5] * 0.03).astype(np.float32)
    transition_np = rng.integers(0, TRANSITION_CLASSES, size=(128,), dtype=np.int64)
    next_x_np = (x_np + rng.normal(scale=0.1, size=x_np.shape)).astype(np.float32)
    terminal_np = np.zeros((128,), dtype=np.float32)
    pad_index = len(x_np)
    history_index_np = np.full((len(x_np), WINDOW_SIZE), pad_index, dtype=np.int64)
    history_mask_np = np.zeros((len(x_np), WINDOW_SIZE), dtype=np.float32)
    for i in range(len(x_np)):
        context = list(range(max(0, i - WINDOW_SIZE + 1), i + 1))
        history_index_np[i, -len(context):] = context
        history_mask_np[i, -len(context):] = 1.0
else:
    rows = load_jsonl(DATA_PATH)
    x_np, y_np, transition_np, next_x_np, terminal_np, history_index_np, history_mask_np = build_training_arrays(rows)
    print(f"loaded {len(rows)} rows from {DATA_PATH}")

feature_mean_np = x_np.mean(axis=0, keepdims=True)
feature_std_np = x_np.std(axis=0, keepdims=True) + 1e-6
x_norm_np = (x_np - feature_mean_np) / feature_std_np
next_x_norm_np = (next_x_np - feature_mean_np) / feature_std_np

x = torch.tensor(x_norm_np, device=DEVICE)
x_padded = torch.cat([x, torch.zeros((1, x.shape[1]), device=DEVICE, dtype=x.dtype)], dim=0)
y = torch.tensor(y_np, device=DEVICE)
transition_target = torch.tensor(transition_np, device=DEVICE)
next_x = torch.tensor(next_x_norm_np, device=DEVICE)
terminal = torch.tensor(terminal_np, device=DEVICE)
history_index = torch.tensor(history_index_np, device=DEVICE)
history_mask = torch.tensor(history_mask_np, device=DEVICE)

print("x", tuple(x.shape), "value", tuple(y.shape), "transition", tuple(transition_target.shape))
print("history", tuple(history_index.shape), "window", WINDOW_SIZE)


loaded 107582 rows from /home/inzi/poke-bot-agent/data/mac-rollouts-100k-fullstate.jsonl
x (107582, 266) value (107582,) transition (107582,)
history (107582, 128) window 128


In [41]:
class TransformerRLModel(torch.nn.Module):
    def __init__(
        self,
        input_dim: int,
        policy_dim: int,
        d_model: int,
        nhead: int,
        num_layers: int,
        dim_feedforward: int,
        dropout: float,
        window_size: int,
    ):
        super().__init__()
        self.input_dim = input_dim
        self.window_size = window_size
        self.token_proj = torch.nn.Linear(input_dim, d_model)
        self.position_embed = torch.nn.Embedding(window_size, d_model)
        encoder_layer = torch.nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True,
            activation="gelu",
        )
        self.encoder = torch.nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.norm = torch.nn.LayerNorm(d_model)
        self.value_head = torch.nn.Sequential(
            torch.nn.Linear(d_model, d_model),
            torch.nn.GELU(),
            torch.nn.Linear(d_model, 1),
        )
        self.policy_head = torch.nn.Sequential(
            torch.nn.Linear(d_model, d_model),
            torch.nn.GELU(),
            torch.nn.Linear(d_model, policy_dim),
        )
        self.next_feature_head = torch.nn.Sequential(
            torch.nn.Linear(d_model, d_model),
            torch.nn.GELU(),
            torch.nn.Linear(d_model, input_dim),
        )
        self.uncertainty_head = torch.nn.Sequential(
            torch.nn.Linear(d_model, d_model),
            torch.nn.GELU(),
            torch.nn.Linear(d_model, 1),
        )

    def encode(self, x: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        positions = torch.arange(x.shape[1], device=x.device)
        tokens = self.token_proj(x) + self.position_embed(positions).unsqueeze(0)
        padding_mask = mask <= 0
        encoded = self.encoder(tokens, src_key_padding_mask=padding_mask)
        return self.norm(encoded[:, -1])

    def forward(self, x: torch.Tensor, mask: torch.Tensor) -> dict[str, torch.Tensor]:
        pooled = self.encode(x, mask)
        return {
            "value": self.value_head(pooled).squeeze(-1),
            "policy_logits": self.policy_head(pooled),
            "next_features": self.next_feature_head(pooled),
            "log_variance": self.uncertainty_head(pooled).squeeze(-1).clamp(-5.0, 5.0),
        }


MODEL_D_MODEL = CONFIG["model"]["d_model"]
MODEL_HEADS = CONFIG["model"]["heads"]
MODEL_LAYERS = CONFIG["model"]["layers"]
MODEL_FF = CONFIG["model"]["ff"]
MODEL_DROPOUT = CONFIG["model"]["dropout"]
LEARNING_RATE = CONFIG["model"]["learning_rate"]
WEIGHT_DECAY = CONFIG["model"]["weight_decay"]
WINDOW_SIZE = CONFIG["window_size"]

if MODEL_D_MODEL % MODEL_HEADS != 0:
    raise ValueError("MODEL_D_MODEL must be divisible by MODEL_HEADS")

model = TransformerRLModel(
    x.shape[1],
    TRANSITION_CLASSES,
    d_model=MODEL_D_MODEL,
    nhead=MODEL_HEADS,
    num_layers=MODEL_LAYERS,
    dim_feedforward=MODEL_FF,
    dropout=MODEL_DROPOUT,
    window_size=WINDOW_SIZE,
).to(DEVICE)
print(
    f"model: d_model={MODEL_D_MODEL} heads={MODEL_HEADS} "
    f"layers={MODEL_LAYERS} ff={MODEL_FF} dropout={MODEL_DROPOUT} window={WINDOW_SIZE}"
)
print(f"parameters: {sum(p.numel() for p in model.parameters()):,}")
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
value_loss_fn = torch.nn.MSELoss()
policy_loss_fn = torch.nn.CrossEntropyLoss()
dynamics_loss_fn = torch.nn.SmoothL1Loss(reduction="none")

VALUE_WEIGHT = CONFIG["loss"]["value"]
POLICY_WEIGHT = CONFIG["loss"]["policy"]
DYNAMICS_WEIGHT = CONFIG["loss"]["dynamics"]
ENTROPY_WEIGHT = CONFIG["loss"]["entropy"]
UNCERTAINTY_WEIGHT = CONFIG["loss"]["uncertainty"]

EPOCHS = CONFIG["training"]["epochs"]
PATIENCE = CONFIG["training"]["patience"]
MIN_DELTA = CONFIG["training"]["min_delta"]
PRINT_EVERY = CONFIG["training"]["print_every"]
BATCH_SIZE = CONFIG["training"]["batch_size"]
NUM_ROWS = int(x.shape[0])
NUM_BATCHES = max(1, (NUM_ROWS + BATCH_SIZE - 1) // BATCH_SIZE)

best_loss = float("inf")
best_epoch = 0
best_state = None
epochs_without_improvement = 0
last_metrics = {}
stopped_early = False
completed_epochs = 0

print(f"training batches: rows={NUM_ROWS} batch_size={BATCH_SIZE} batches={NUM_BATCHES}")

progress = tqdm(range(EPOCHS), desc="training", unit="epoch")
for epoch in progress:
    order = torch.randperm(NUM_ROWS, device=DEVICE)
    metric_sums = {
        "total_loss": 0.0,
        "value_loss": 0.0,
        "policy_loss": 0.0,
        "dynamics_loss": 0.0,
        "entropy": 0.0,
        "uncertainty_loss": 0.0,
    }
    seen = 0

    batch_progress = tqdm(
        range(0, NUM_ROWS, BATCH_SIZE),
        desc=f"epoch {epoch + 1}/{EPOCHS} batches",
        unit="batch",
        total=NUM_BATCHES,
        leave=False,
    )
    for batch_number, start in enumerate(batch_progress, start=1):
        batch_idx = order[start:start + BATCH_SIZE]
        xb = x_padded[history_index[batch_idx]]
        mask_b = history_mask[batch_idx]
        yb = y[batch_idx]
        transition_b = transition_target[batch_idx]
        next_xb = next_x[batch_idx]
        terminal_b = terminal[batch_idx]

        optimizer.zero_grad(set_to_none=True)
        out = model(xb, mask_b)

        value_loss = value_loss_fn(out["value"], yb)
        policy_loss = policy_loss_fn(out["policy_logits"], transition_b)
        nonterminal = (1.0 - terminal_b).unsqueeze(-1)
        dynamics_loss = (dynamics_loss_fn(out["next_features"], next_xb) * nonterminal).sum() / nonterminal.sum().clamp_min(1.0)
        probs = torch.softmax(out["policy_logits"], dim=-1)
        entropy = -(probs * torch.log(probs.clamp_min(1e-8))).sum(dim=-1).mean()
        uncertainty_loss = torch.mean(torch.exp(-out["log_variance"]) * (out["value"] - yb).pow(2) + out["log_variance"])

        loss = (
            VALUE_WEIGHT * value_loss
            + POLICY_WEIGHT * policy_loss
            + DYNAMICS_WEIGHT * dynamics_loss
            - ENTROPY_WEIGHT * entropy
            + UNCERTAINTY_WEIGHT * uncertainty_loss
        )
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        batch_size_actual = int(xb.shape[0])
        seen += batch_size_actual
        metric_sums["total_loss"] += float(loss.detach().cpu()) * batch_size_actual
        metric_sums["value_loss"] += float(value_loss.detach().cpu()) * batch_size_actual
        metric_sums["policy_loss"] += float(policy_loss.detach().cpu()) * batch_size_actual
        metric_sums["dynamics_loss"] += float(dynamics_loss.detach().cpu()) * batch_size_actual
        metric_sums["entropy"] += float(entropy.detach().cpu()) * batch_size_actual
        metric_sums["uncertainty_loss"] += float(uncertainty_loss.detach().cpu()) * batch_size_actual
        batch_progress.set_postfix({
            "batch": f"{batch_number}/{NUM_BATCHES}",
            "loss": f"{float(loss.detach().cpu()):.5f}",
            "v": f"{float(value_loss.detach().cpu()):.4f}",
            "p": f"{float(policy_loss.detach().cpu()):.4f}",
            "dyn": f"{float(dynamics_loss.detach().cpu()):.4f}",
        })

    loss_value = metric_sums["total_loss"] / max(1, seen)
    last_metrics = {
        name: total / max(1, seen)
        for name, total in metric_sums.items()
    }
    completed_epochs = epoch + 1
    if loss_value < best_loss - MIN_DELTA:
        best_loss = loss_value
        best_epoch = epoch + 1
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    progress.set_postfix({
        "loss": f"{loss_value:.5f}",
        "v": f"{last_metrics['value_loss']:.4f}",
        "p": f"{last_metrics['policy_loss']:.4f}",
        "dyn": f"{last_metrics['dynamics_loss']:.4f}",
        "best": f"{best_loss:.5f}@{best_epoch}",
        "patience": f"{epochs_without_improvement}/{PATIENCE}",
    })

    if epochs_without_improvement >= PATIENCE:
        progress.set_postfix({
            "loss": f"{loss_value:.5f}",
            "best": f"{best_loss:.5f}@{best_epoch}",
            "patience": f"{epochs_without_improvement}/{PATIENCE}",
            "stopped": "early",
        })
        print(f"early stopping at epoch={epoch + 1}; best={best_loss:.5f}@{best_epoch}")
        stopped_early = True
        break
progress.close()

if best_state is not None:
    model.load_state_dict(best_state)

training_report = {
    "completed_epochs": completed_epochs,
    "requested_epochs": EPOCHS,
    "stopped_early": stopped_early,
    "best_total_loss": best_loss,
    "best_epoch": best_epoch,
    "last_metrics": last_metrics,
    "dataset_rows": int(x.shape[0]),
    "input_dim": int(x.shape[1]),
    "window_size": WINDOW_SIZE,
    "batch_size": BATCH_SIZE,
    "device": str(DEVICE),
    "data_path": str(DATA_PATH) if DATA_PATH else None,
    "loss_note": "Loss is the training objective, not winrate. Winrate requires CABT evaluation games using the model as the action policy.",
}


model: d_model=512 heads=8 layers=8 ff=2048 dropout=0.1 window=128
parameters: 26,614,548
training batches: rows=107582 batch_size=256 batches=421


training:   0%|          | 0/500 [00:00<?, ?epoch/s]

epoch 1/500 batches:   0%|          | 0/421 [00:00<?, ?batch/s]

epoch 2/500 batches:   0%|          | 0/421 [00:00<?, ?batch/s]

epoch 3/500 batches:   0%|          | 0/421 [00:00<?, ?batch/s]

...

epoch 287/500 batches:   0%|          | 0/421 [00:00<?, ?batch/s]

epoch 288/500 batches:   0%|          | 0/421 [00:00<?, ?batch/s]

epoch 289/500 batches:   0%|          | 0/421 [00:00<?, ?batch/s]

epoch 290/500 batches:   0%|          | 0/421 [00:00<?, ?batch/s]

early stopping at epoch=290; best=1.54387@280


In [42]:
OUT = ROOT / "out/value_model.pt"
OUT.parent.mkdir(parents=True, exist_ok=True)

COMPETITION_RESULTS_PATH = CONFIG["competition_results_path"]

def load_competition_results(path: Path) -> list[dict]:
    if not path.exists():
        return []
    with path.open("r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

competition_results = load_competition_results(COMPETITION_RESULTS_PATH)
latest_competition_result = competition_results[0] if competition_results else None
training_report["competition_results_path"] = str(COMPETITION_RESULTS_PATH)
training_report["latest_competition_result"] = latest_competition_result

checkpoint = {
    "model_state_dict": model.state_dict(),
    "model_type": "temporal_transformer_rl_complex_loss",
    "input_dim": x.shape[1],
    "policy_dim": TRANSITION_CLASSES,
    "model_config": {
        "d_model": MODEL_D_MODEL,
        "heads": MODEL_HEADS,
        "layers": MODEL_LAYERS,
        "dim_feedforward": MODEL_FF,
        "dropout": MODEL_DROPOUT,
        "window_size": WINDOW_SIZE,
    },
    "feature_mean": feature_mean_np.astype(np.float32).tolist(),
    "feature_std": feature_std_np.astype(np.float32).tolist(),
    "loss_weights": {
        "value": VALUE_WEIGHT,
        "policy": POLICY_WEIGHT,
        "dynamics": DYNAMICS_WEIGHT,
        "entropy": ENTROPY_WEIGHT,
        "uncertainty": UNCERTAINTY_WEIGHT,
    },
    "training_report": training_report,
    "device_used": str(DEVICE),
    "data_path": str(DATA_PATH) if DATA_PATH else None,
}
torch.save(checkpoint, OUT)

print("saved", OUT)
print("\nFinal training report")
print("-" * 22)
print(f"rows: {training_report['dataset_rows']}")
print(f"window: {training_report['window_size']} batch: {training_report['batch_size']}")
print(f"device: {training_report['device']}")
print(f"epochs: {training_report['completed_epochs']} / {training_report['requested_epochs']}")
print(f"early stopped: {training_report['stopped_early']}")
print(f"best total loss: {training_report['best_total_loss']:.5f} @ epoch {training_report['best_epoch']}")
for name, value in training_report["last_metrics"].items():
    print(f"{name}: {value:.5f}")
if latest_competition_result:
    print("\nLatest official Kaggle result")
    print("-" * 29)
    print(f"file: {latest_competition_result['file_name']}")
    print(f"date: {latest_competition_result['date']}")
    print(f"status: {latest_competition_result['status']}")
    print(f"public score: {latest_competition_result['public_score']}")
    print(f"private score: {latest_competition_result['private_score']}")
else:
    print(f"\nNo official Kaggle results found at {COMPETITION_RESULTS_PATH}")
print("\nInterpretation")
print("- Loss is not winrate.")
print("- Lower value_loss means better win/loss prediction on rollout states.")
print("- Official Kaggle score comes from scripts/fetch_competition_results.py after a submission finishes.")


saved /home/inzi/poke-bot-agent/out/value_model.pt

Final training report
----------------------
rows: 107582
window: 128 batch: 256
device: cuda
epochs: 290 / 500
early stopped: True
best total loss: 1.54387 @ epoch 280
total_loss: 1.54428
value_loss: 0.02368
policy_loss: 0.08027
dynamics_loss: 10.57894
entropy: 0.09089
uncertainty_loss: -4.67129

No official Kaggle results found at /home/inzi/poke-bot-agent/data/competition-results.jsonl

Interpretation
- Loss is not winrate.
- Lower value_loss means better win/loss prediction on rollout states.
- Official Kaggle score comes from scripts/fetch_competition_results.py after a submission finishes.
